# Lecture 7 · Exercises

Code for [exercises.md](exercises.md)

In [ ]:
from math import comb
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

DATA = Path("../../coding_tasks/Data/Out_Data")

---

## Exercise 7.1 · Flu shot experiment

Table reading, no dataset. The two checks worth doing are the relative effects and the implied t-values, neither of which the table prints.

In [ ]:
flu = pd.DataFrame(
    {
        "effect_pp": [2.4028, 2.3196],
        "se": [0.3447, 1.3611],
        "baseline_pct": [7.034, 67.754],
        "n": [24_270, 4_595],
    },
    index=["no flu shot in previous year", "vaccinated in previous year"],
)
flu

In [ ]:
flu["relative_pct"] = 100 * flu["effect_pp"] / flu["baseline_pct"]
flu["t"] = flu["effect_pp"] / flu["se"]
flu["p"] = 2 * (1 - stats.norm.cdf(flu["t"].abs()))
flu.round(3)

Nearly identical absolute effects, relative effects a factor of ten apart. The baseline does that, not the behaviour: two thirds of the previously vaccinated were going to get the shot anyway, so there is little room left for a letter to move.

---

## Exercise 7.2 · Working from home

In [ ]:
home = pd.read_csv(DATA / "lecture7_working_from_home.csv")
print(home.shape)
print("missing values:", home.isna().sum().to_dict())
home.head()

### 1 · the outcome variable

In [ ]:
print("distinct values of attrite_perc:", sorted(home["attrite_perc"].unique()))

Binary, scaled to percent. A group mean reads as an attrition rate in percent, a regression coefficient as an effect in percentage points.

### 2 · mean comparison

In [ ]:
means = home.groupby("treat")["attrite_perc"].agg(["mean", "size"])
print(means.round(4))
print(f"\ntreatment effect = {means.loc[1, 'mean'] - means.loc[0, 'mean']:.4f} percentage points")
print(f"relative         = {(means.loc[1, 'mean'] / means.loc[0, 'mean'] - 1):.1%} of the control rate")

### 3, 4, 6 · regressions, overall and by subgroup

In [ ]:
def effect(data, label):
    r = smf.ols("attrite_perc ~ treat", data=data).fit()
    return {
        "subgroup": label,
        "n": int(r.nobs),
        "control": r.params["Intercept"],
        "treatment": r.params["Intercept"] + r.params["treat"],
        "effect": r.params["treat"],
        "se": r.bse["treat"],
        "p": r.pvalues["treat"],
    }


rows = [
    effect(home, "all"),
    effect(home[home["role"] == 1], "managers"),
    effect(home[home["role"] == 0], "non-managers"),
    effect(home[home["male"] == 0], "women"),
    effect(home[home["male"] == 1], "men"),
    effect(home[home["live_far"] == 1], "commute > 90 min"),
    effect(home[home["live_far"] == 0], "commute <= 30 min"),
]
pd.DataFrame(rows).set_index("subgroup").round(3)

### 5 · comparing with the paper

Every point estimate and p-value matches the bar chart in Bloom, Han and Liang (2024). Their figure is a plot of exactly these group means.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.6), sharey=True)
panels = [("all", None), ("non-managers", "managers"), ("women", "men")]
by_name = {r["subgroup"]: r for r in rows}

for ax, (left, right) in zip(axes, panels):
    labels = [left] if right is None else [left, right]
    x = 0
    for name in labels:
        r = by_name[name]
        for value, colour in [(r["control"], "#1f4e79"), (r["treatment"], "#e84d22")]:
            ax.bar(x, value, width=0.95, color=colour)
            ax.text(x, value + 0.2, f"{value:.1f}", ha="center", fontsize=8)
            x += 1
        top = max(r["control"], r["treatment"]) + 1.3
        ax.plot([x - 2, x - 1], [top, top], color="#444", linewidth=0.8)
        ax.text(x - 1.5, top + 0.15, f"P = {r['p']:.3g}", ha="center", fontsize=8)
        x += 0.6
    ax.set_xticks([])
    ax.set_title(" / ".join(labels), fontsize=9)
    ax.set_ylim(0, 11)
    ax.spines[["top", "right"]].set_visible(False)

axes[0].set_ylabel("Attrition (%)")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in ["#1f4e79", "#e84d22"]]
fig.legend(handles, ["In-person (control)", "Hybrid WFH (treatment)"],
           loc="lower center", ncol=2, frameon=False, fontsize=9)
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

### Are the subgroups actually different?

Comparing a significant subgroup against a non-significant one is not a test. The question has its own estimate and standard error: the interaction term.

In [ ]:
inter = []
for v in ["role", "male", "live_far"]:
    r = smf.ols(f"attrite_perc ~ treat * {v}", data=home).fit()
    k = f"treat:{v}"
    inter.append({"interaction": k, "n": int(r.nobs), "difference in effect": r.params[k],
                  "se": r.bse[k], "p": r.pvalues[k]})
pd.DataFrame(inter).set_index("interaction").round(3)

None of the three is significant. The commute split is the clearest case: the two point estimates differ by half a percentage point with p = 0.85, yet one crossed the 10% threshold and the other did not.

**The difference between "significant" and "not significant" is not itself statistically significant.** To claim a differential effect, test the difference.

### Was the randomization balanced?

The lecture 6 check, on the three covariates available.

In [ ]:
pd.DataFrame(
    [
        {
            "covariate": v,
            "difference": smf.ols(f"{v} ~ treat", data=home).fit().params["treat"],
            "p": smf.ols(f"{v} ~ treat", data=home).fit().pvalues["treat"],
        }
        for v in ["role", "male", "live_far"]
    ]
).set_index("covariate").round(4)

The odd-birthday rule did its job.

The outcome is binary, so the errors are heteroskedastic by construction and robust standard errors are the safer default. Worth checking rather than assuming.

In [ ]:
classical = smf.ols("attrite_perc ~ treat", data=home).fit()
robust = smf.ols("attrite_perc ~ treat", data=home).fit(cov_type="HC1")
print(f"classical: se = {classical.bse['treat']:.4f}, p = {classical.pvalues['treat']:.4f}")
print(f"robust:    se = {robust.bse['treat']:.4f}, p = {robust.pvalues['treat']:.4f}")

---

## Beyond the exercise · the lecture's own examples

### Example 7.2 and 7.4 · mean comparison, regression and t-test are one thing

In [ ]:
ex = pd.read_csv(DATA / "lecture5_experimental_data.csv")
ex.groupby("D")["Y"].agg(["mean", "var", "size"]).round(4)

In [ ]:
t_grp, c_grp = ex.loc[ex["D"] == 1, "Y"], ex.loc[ex["D"] == 0, "Y"]

t_by_hand = (t_grp.mean() - c_grp.mean()) / np.sqrt(t_grp.var() / len(t_grp) + c_grp.var() / len(c_grp))
reg = smf.ols("Y ~ D", data=ex).fit()

print(f"t by hand           {t_by_hand:.5f}")
print(f"t-test (pooled)     {stats.ttest_ind(t_grp, c_grp, equal_var=True).statistic:.5f}"
      f"   p = {stats.ttest_ind(t_grp, c_grp, equal_var=True).pvalue:.4f}  df = 6")
print(f"t-test (Welch)      {stats.ttest_ind(t_grp, c_grp, equal_var=False).statistic:.5f}"
      f"   p = {stats.ttest_ind(t_grp, c_grp, equal_var=False).pvalue:.4f}  df = 5.55")
print(f"regression on D     {reg.tvalues['D']:.5f}   p = {reg.pvalues['D']:.4f}")

Pooled and Welch give the same t here because the groups are the same size, which makes the two formulas algebraically identical. They differ only in the degrees of freedom, so the p-values differ slightly.

In [ ]:
lo, hi = reg.conf_int().loc["D"]
print(f"estimated effect: {reg.params['D']:.1f}")
print(f"95% CI:           [{lo:.1f}, {hi:.1f}]")
print(f"true effect in the data: {(ex['Y1'] - ex['Y0']).mean():.1f}")

Every student in this dataset really is helped by exactly 10 points, the selection bias is exactly zero, and the experiment still cannot tell the effect apart from zero. The interval runs from strongly harmful to strongly helpful.

A perfect design with n = 8 buys you an unbiased estimate and nothing else. That gap is what lecture 11 is for.

### Example 7.1 · Lady tasting tea

Eight cups, four with milk first. Under pure guessing every one of the C(8,4) = 70 ways to pick the four milk-first cups is equally likely, so the null distribution is exact and needs no approximation.

In [ ]:
total = comb(8, 4)
tea = pd.DataFrame(
    {
        "correct": range(5),
        "ways": [comb(4, k) * comb(4, 4 - k) for k in range(5)],
    }
)
tea["probability"] = tea["ways"] / total
print(f"C(8,4) = {total}, so the p-value for all four correct is 1/{total} = {1 / total:.4f}")
tea.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(tea["correct"], tea["probability"], width=0.6, color="#4C78A8")
ax.bar([4], tea.loc[4, "probability"], width=0.6, color="#B03A2E")
ax.set_xlabel("number of successes (red: all four correct, the claim being tested)")
ax.set_ylabel("probability")
ax.set_xticks(range(5))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

The distribution is symmetric: getting none right is as unlikely as getting all right, because placing all four in the wrong group is perfect discrimination backwards.

This is randomization inference, and the lecture 5 notebook already did it without the name. Enumerating all 35 ways to treat 3 of 7 students is the same procedure. The classical t-test above is the large-sample approximation to this idea, for when the assignments are too many to count.

One design consequence: with eight cups the smallest reachable p-value is 1/70 = 0.014. Six cups would cap it at 1/20 = 0.05, so no performance however perfect could clear the 1% level. The design limits what the result can show, before any data exist.

In [ ]:
for n_cups in [6, 8, 10, 12]:
    half = n_cups // 2
    print(f"{n_cups:2d} cups: C({n_cups},{half}) = {comb(n_cups, half):4d} "
          f"-> smallest possible p-value = {1 / comb(n_cups, half):.4f}")